In [2]:
from cad_recon_lib import (
    ABCMultiModalDataset,
    ReconstructionOptions,
    visualize_multimodal_sample,
    visualize_brep_reconstruction_comparison,
)

import torch
from cad_recon_network import CADReconDualBackbone

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
dataset = ABCMultiModalDataset(base_dir="./abc_dataset_filtered-1",
                               pcd_num_points=2048, voxel_res=64)

OBJ file scanning...
144153 OBJ files loaded.


### random data sample vis.

In [4]:
import random

# sample = dataset[random.randint(0, len(dataset)-1)]
sample = dataset[301]
visualize_multimodal_sample(dataset, sample=sample)

Visualizing Model ID: 00000574
Rendered:
 - LEFT  : STEP B-Rep (primitive color + black wireframe)
 - MIDDLE: OBJ mesh (gray) + simulated PCD (red)
 - RIGHT : voxel grid (green)


In [ ]:
opts = ReconstructionOptions(fast_vis_mode=True)
visualize_brep_reconstruction_comparison(sample, options=opts)

### backbone check

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

if "sample" not in globals():
    sample = dataset[0]

pcd = sample["pcd"].unsqueeze(0).to(device)      # [B, N, 3]
voxel = sample["voxel"].unsqueeze(0).to(device)  # [B, 1, R, R, R]
voxel_res = int(voxel.shape[-1])

model = CADReconDualBackbone(
    pointnet_kwargs={"variant": "msg", "use_normals": False, "feature_dim": 1024},
    sparseconv_kwargs={
        "voxel_resolution": voxel_res,
        "in_channels": 1,
        "feature_dim": 256,
        "allow_fallback": False,
    },
).to(device).eval()

with torch.no_grad():
    out = model(pcd=pcd, voxel=voxel)

print("pcd input:", tuple(pcd.shape), pcd.device)
print("voxel input:", tuple(voxel.shape), voxel.device)
print("pcd_feature:", tuple(out["pcd_feature"].shape), out["pcd_feature"].device)
print("voxel_feature:", tuple(out["voxel_feature"].shape), out["voxel_feature"].device)
print("fused_feature:", tuple(out["fused_feature"].shape), out["fused_feature"].device)


### training command (script)

`cad_recon_network/scripts/train_brep_model.py`


In [ ]:
import shlex
import subprocess

python_exe = r"C:\Users\ywchoi\.conda\envs\cadgen\python.exe"
cmd = [
    python_exe,
    "cad_recon_network/scripts/train_brep_model.py",
    "--base-dir", "./abc_dataset_filtered-1",
    "--output-dir", "cad_recon_network/weights/brep_runs/exp_debug",
    "--device", "cuda" if torch.cuda.is_available() else "cpu",
    "--epochs", "50",
    "--batch-size", "1",
    "--num-workers", "4",
    "--pointnet-variant", "msg",
]

print(" ".join(shlex.quote(x) for x in cmd))

### inline mini training example (5 steps)

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
from cad_recon_network import CADReconBRepModel
from cad_recon_network.scripts.train_brep_model import cad_collate_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

subset_size = min(8, len(dataset))
subset = Subset(dataset, list(range(subset_size)))
loader = DataLoader(subset, batch_size=1, shuffle=True, num_workers=0, collate_fn=cad_collate_fn)

model = CADReconBRepModel(
    pointnet_kwargs={"variant": "msg", "use_normals": False, "feature_dim": 1024},
    sparseconv_kwargs={
        "voxel_resolution": 64,
        "in_channels": 1,
        "feature_dim": 256,
        "allow_fallback": False,
    },
    head_kwargs={
        "max_vertices": dataset.max_vertices,
        "max_edges": dataset.max_edges,
        "max_faces": dataset.max_faces,
        "hidden_dim": 256,
        "num_decoder_layers": 2,
        "num_curve_types": 9,
        "num_surface_types": 11,
    },
).to(device)

# batch_size=1?? PointNet BN ???? ?? eval ??
for m in model.backbone.pointnet.model.modules():
    if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
        m.eval()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
loss_weights = {
    "v_feat": 1.0, "e_feat": 1.0, "f_feat": 1.0,
    "e_type": 0.5, "f_type": 0.5, "e_ori": 0.25, "f_ori": 0.25,
    "adj_ev": 0.5, "adj_fe": 0.5, "counts": 0.2,
}

model.train()
for step, batch in enumerate(loader, start=1):
    for k, v in list(batch.items()):
        if torch.is_tensor(v):
            batch[k] = v.to(device, non_blocking=True)

    optimizer.zero_grad(set_to_none=True)
    _, loss_dict = model.forward_with_loss(batch, normalize_targets=True, loss_weights=loss_weights)
    loss = loss_dict["total"]
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    print(
        f"step={step:02d} total={loss_dict["total"].item():.4f} "
        f"v={loss_dict["v_feat"].item():.4f} e={loss_dict["e_feat"].item():.4f} f={loss_dict["f_feat"].item():.4f} "
        f"e_type={loss_dict["e_type"].item():.4f} f_type={loss_dict["f_type"].item():.4f}"
    )

    if step >= 5:
        break
